In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_mamlpp_MOE_HPO_DeepCNNLSTM_V2"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_mamlpp_MOE_HPO_DeepCNNLSTM_V2
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\1s3w_mamlpp_MOE_HPO_DeepCNNLSTM_V2.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'mamlpp_MOE_PAPER_DeepCNNLSTM_1fcv_hpo'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 192 trials.
Best value (Accuracy): 0.9479
Best Hyperparameters:
  maml_inner_steps: 20
  maml_inner_steps_eval: 100
  maml_alpha_init: 0.004281682811626557
  maml_alpha_init_eval: 0.0358651952640868
  outer_lr: 0.00016261943856970884
  wd: 0.0007654571626051887
  groupnorm_num_groups: 4
  num_experts: 40
  MOE_top_k: 10
  MOE_gate_temperature: 0.3201231267680839
  MOE_aux_coeff: 0.06793873732862875
  MOE_aux_loss_plcmt: both
  episodes_per_epoch_train: 150
  label_smooth: 0.15
  use_lslr_at_eval: True


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [12]:
# FULL
params_to_plot = [
  "maml_inner_steps", "maml_inner_steps_eval", "maml_alpha_init", "maml_alpha_init_eval", "outer_lr", "wd", 
  "groupnorm_num_groups", "num_experts", "MOE_top_k", "MOE_gate_temperature", "MOE_aux_coeff", "MOE_aux_loss_plcmt", 
  "episodes_per_epoch_train", "label_smooth", "use_lslr_at_eval"]

params_to_plot_OPT = [
    "outer_lr", "wd", "groupnorm_num_groups", "episodes_per_epoch_train", "label_smooth",
]

params_to_plot_MAML = [
    "maml_inner_steps", "maml_inner_steps_eval", "maml_alpha_init", "maml_alpha_init_eval","use_lslr_at_eval",
]

params_to_plot_MOE_CORE = [
    "num_experts", "MOE_top_k", "MOE_gate_temperature", "MOE_aux_coeff", "MOE_aux_loss_plcmt"
]


In [13]:
from optuna.visualization import plot_slice


In [14]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [15]:
fig_slice = plot_slice(study, params=params_to_plot_MAML)
fig_slice.show()

In [16]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_CORE)
fig_slice.show()

## Best Hyperparameter Ranges



In [17]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
110,110,0.947917,0.939583,2026-04-10 04:54:42.587444,0 days 02:03:12.356093
111,111,0.947917,0.939583,2026-04-10 04:55:08.465143,0 days 02:03:21.810989
112,112,0.947917,0.939583,2026-04-10 04:58:32.297172,0 days 02:04:00.123668
119,119,0.942708,0.937500,2026-04-10 05:52:07.813494,0 days 01:03:38.511475
120,120,0.942708,0.937500,2026-04-10 05:52:23.046562,0 days 01:03:39.022753
99,99,0.941667,0.933333,2026-04-10 02:41:39.354031,0 days 01:29:20.557004
100,100,0.941667,0.933333,2026-04-10 02:46:34.388694,0 days 01:29:50.945594
138,138,0.941667,0.943750,2026-04-10 07:28:15.446380,0 days 01:29:31.789656
115,115,0.938542,0.937500,2026-04-10 05:46:55.429218,0 days 01:42:49.319854
117,117,0.938542,0.937500,2026-04-10 05:49:27.118140,0 days 01:38:48.212018


In [18]:
from optuna.trial import TrialState

# 1. Filter for only successfully completed trials
completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

# 2. Sort the filtered list (using reverse=True if you want the highest values)
top_trials = sorted(completed_trials, key=lambda t: t.value, reverse=True)[:10]

# 3. Print the params
for t in top_trials:
    print(t.params)

{'maml_inner_steps': 20, 'maml_inner_steps_eval': 100, 'maml_alpha_init': 0.004281682811626557, 'maml_alpha_init_eval': 0.0358651952640868, 'outer_lr': 0.00016261943856970884, 'wd': 0.0007654571626051887, 'groupnorm_num_groups': 4, 'num_experts': 40, 'MOE_top_k': 10, 'MOE_gate_temperature': 0.3201231267680839, 'MOE_aux_coeff': 0.06793873732862875, 'MOE_aux_loss_plcmt': 'both', 'episodes_per_epoch_train': 150, 'label_smooth': 0.15, 'use_lslr_at_eval': True}
{'maml_inner_steps': 20, 'maml_inner_steps_eval': 100, 'maml_alpha_init': 0.004281682811626557, 'maml_alpha_init_eval': 0.0358651952640868, 'outer_lr': 0.00016261943856970884, 'wd': 0.0007654571626051887, 'groupnorm_num_groups': 4, 'num_experts': 40, 'MOE_top_k': 10, 'MOE_gate_temperature': 0.3201231267680839, 'MOE_aux_coeff': 0.06793873732862875, 'MOE_aux_loss_plcmt': 'both', 'episodes_per_epoch_train': 150, 'label_smooth': 0.15, 'use_lslr_at_eval': True}
{'maml_inner_steps': 20, 'maml_inner_steps_eval': 100, 'maml_alpha_init': 0.00